### 0. Environment Setup

In [ ]:
!nvidia-smi

Wed Nov 18 08:09:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU Name:", torch.cuda.get_device_name(0))
  print("GPU Memory:", round(
      torch.cuda.get_device_properties(0).total_memory / 1024**3 , 2
      ), "GB")
  print("BF16 Supported:", torch.cuda.is_bf16_supported())

PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB
BF16 Supported: True


In [1]:
!pip install unsloth

In [ ]:
!unsloth install-kernels xformers

In [ ]:
import unsloth
import torch

print("Unsloth Version:", unsloth.__version__)
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1118 08:11:08.778000 374 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1118 08:11:08.840000 374 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth Version: 2026.11.6
PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU: Tesla T4


### 1. Understanding Qwen3-4B
###
###

| Parameter | Value |
|---|---|
| Architecture | Dense Transformer |
| Parameters | 4 Billion |
| Layers | 36 |
| Attention Heads | 32 |
| KV Heads | 8 |
| Native Context Length | 32,768 |
| Vocabulary | حدود 151K |
| License | Apache 2.0 |
| Original Weight Format | BF16 |

In [ ]:
import torch
import transformers
import unsloth

print("========== Environment ==========")

print(f"PyTorch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"Unsloth Version: {unsloth.__version__}")

print("\n========== GPU Information ==========")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)

    print(f"GPU: {gpu.name}")
    print(f"Total VRAM: {gpu.total_memory / 1024**3:.2f} GB")
    print(f"CUDA Capability: {gpu.major}.{gpu.minor}")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")
    print(f"FP16 Supported: True")
else:
    print("CUDA is not available!")

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:2279: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1121 13:51:06.192000 879 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1121 13:51:06.244000 879 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
========== Environment ==========
PyTorch Version: 2.14.1+cu130
Transformers Version: 5.22.1
Unsloth Version: 2026.11.9

========== GPU Information ==========
GPU: Tesla T4
Total VRAM: 14.56 GB
CUDA Capability: 7.5
BF16 Supported: False
FP16 Supported: True


In [2]:
from transformers import AutoConfig, AutoTokenizer

model_name = "Qwen/Qwen3-4B"

print("Loading model configuration...")

config = AutoConfig.from_pretrained(model_name)

print("\n========== Model Configuration ==========")

print(f"Model Type: {config.model_type}")
print(f"Architecture: {config.architectures}")
print(f"Hidden Size: {config.hidden_size}")
print(f"Number of Layers: {config.num_hidden_layers}")
print(f"Attention Heads: {config.num_attention_heads}")
print(f"KV Heads: {config.num_key_value_heads}")
print(f"Vocabulary Size: {config.vocab_size}")
print(f"Maximum Position Embeddings: {config.max_position_embeddings}")

Loading model configuration...



========== Model Configuration ==========
Model Type: qwen3
Architecture: ['Qwen3ForCausalLM']
Hidden Size: 2560
Number of Layers: 36
Attention Heads: 32
KV Heads: 8
Vocabulary Size: 151936
Maximum Position Embeddings: 40960


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("========== Tokenizer Information ==========")

print(f"Tokenizer Class: {tokenizer.__class__.__name__}")
print(f"Vocabulary Size: {len(tokenizer)}")
print(f"EOS Token: {tokenizer.eos_token}")
print(f"PAD Token: {tokenizer.pad_token}")

print("\n========== Chat Template ==========")

print(tokenizer.chat_template[:1500])

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

========== Tokenizer Information ==========
Tokenizer Class: Qwen2Tokenizer
Vocabulary Size: 151669
EOS Token: <|im_end|>
PAD Token: <|endoftext|>

========== Chat Template ==========
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- 

### 2. Baseline Evaluation

In [ ]:
import torch
from transformers import AutoModelForCausalLM

print("Loading Qwen3-4B in FP16...")

model_mame = "Qwen/Qwen3-4B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("\nModel loaded successfully!")

print(f"Model Device: {model.device}")
print(
    f"Allocated VRAM: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)
print(
    f"Reversed VRAM: "
    f"{torch.cuda.memory_reversed() / 1024**3:.2f} GB"
)

Loading Qwen3-4B in FP16...


NameError: name 'model_name' is not defined

### 3. Dataset Engineering

In [56]:
INTENTS = [
    "order_status",
    "shipping_issue",
    "refund_request",
    "return_request",
    "product_information",
    "technical_issue",
    "billing_issue",
    "complaint",
    "other"
]

In [57]:
OUTPUT_SCHEMA = {
    "intent": "string",
    "response": "string",
    "needs_clarification": "boolean"
}

In [58]:
SYSTEM_PROMPT = """
You are SupportMind, a professional customer support assistant.

Your responsibilities:
1. Identify the customer's primary intent.
2. Provide a helpful and professional response.
3. Ask for clarification when required.
4. Never invent information.
5. Focus on the customer's primary requested action
   when selecting an intent.

Supported intents:
- order_status
- shipping_issue
- refund_request
- return_request
- product_information
- technical_issue
- billing_issue
- complaint
- other

Return only a valid JSON object containing exactly:
- intent: string
- response: string
- needs_clarification: boolean

Do not include Markdown or explanations outside JSON.
"""

In [59]:
import json

train_samples = [
    {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": "Where is my order? My order ID is 12345."
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "intent": "order_status",
                    "response": "I can help you check your order. Please allow me a moment to review its status.",
                    "needs_clarification": False
                })
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": "I want a refund."
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "intent": "refund_request",
                    "response": "I can help you with your refund request. Could you provide your order ID?",
                    "needs_clarification": True
                })
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": "I want to return my shoes because they arrived damaged."
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "intent": "return_request",
                    "response": "I'm sorry your shoes arrived damaged. Could you provide your order ID so I can help you with the return?",
                    "needs_clarification": True
                })
            }
        ]
    }
]

In [60]:
def validate_sample(sample):

  messages = sample["messages"]

  assert len(messages) == 3

  assert messages[0]["role"] == "system"
  assert messages[1]["role"] == "user"
  assert messages[2]["role"] == "assistant"

  output = json.loads(messages[2]["content"])

  assert set(output.keys()) == {
      "intent",
      "response",
      "needs_clarification"
  }

  assert output["intent"] in INTENTS
  assert isinstance(output["response"], str)
  assert isinstance(output["needs_clarification"], bool)

  return True


for sample in train_samples:
  validate_sample(sample)

print("All samples are valid!")


All samples are valid!


In [61]:
def make_sample(
    user_message,
    intent,
    response,
    needs_clarification,
    group_id
):
    assistant_output = {
        "intent": intent,
        "response": response,
        "needs_clarification": needs_clarification
    }

    return {
        "group_id": group_id,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            },
            {
                "role": "assistant",
                "content": json.dumps(
                    assistant_output,
                    ensure_ascii=False
                )
            }
        ]
    }

In [62]:
seed_data = [

    # 1. ORDER STATUS

    (
        "Where is my order? My order ID is 12345.",
        "order_status",
        "I can help you with your order inquiry. I don't have direct access to live order information, but you can check the latest status using your order ID on our order tracking page.",
        False,
        "G001"
    ),

    (
        "Can you tell me where my package is?",
        "order_status",
        "I'd be happy to help you check your order. Could you provide your order ID so I can guide you through the tracking process?",
        True,
        "G002"
    ),

    (
        "I placed an order three days ago, but I haven't received any updates.",
        "order_status",
        "I understand your concern. Could you share your order ID so I can help you find the appropriate way to check its status?",
        True,
        "G003"
    ),


    # 2. SHIPPING ISSUE

    (
        "My package is five days late.",
        "shipping_issue",
        "I'm sorry your package is delayed. Could you provide your order ID so I can guide you through the available shipping support options?",
        True,
        "G004"
    ),

    (
        "The tracking information says delivered, but I never received my package.",
        "shipping_issue",
        "I'm sorry you haven't received your package. Could you share your order ID and confirm whether the tracking page shows a delivery time or proof of delivery?",
        True,
        "G005"
    ),

    (
        "My package arrived with damaged packaging.",
        "shipping_issue",
        "I'm sorry your package arrived in that condition. Could you provide your order ID and briefly describe the damage to the packaging?",
        True,
        "G006"
    ),


    # 3. REFUND REQUEST

    (
        "I want a refund for my purchase.",
        "refund_request",
        "I can help you understand the refund process. Could you provide your order ID so I can guide you through the next steps?",
        True,
        "G007"
    ),

    (
        "The product was defective and I want my money back.",
        "refund_request",
        "I'm sorry the product was defective. Could you provide your order ID and briefly describe the defect so I can guide you through the refund process?",
        True,
        "G008"
    ),

    (
        "I received a refund confirmation, but the money hasn't appeared in my account.",
        "refund_request",
        "I understand your concern. Could you share your order ID and the date of the refund confirmation? I can help you understand the next steps for checking the refund.",
        True,
        "G009"
    ),


    # 4. RETURN REQUEST

    (
        "I want to return the shoes I purchased.",
        "return_request",
        "I can help you understand the return process. Could you provide your order ID so I can guide you through the next steps?",
        True,
        "G010"
    ),

    (
        "These shoes are too small. I want to send them back.",
        "return_request",
        "I'm sorry the shoes don't fit. Could you provide your order ID so I can guide you through the return process?",
        True,
        "G011"
    ),

    (
        "I want to return my order because the shoes arrived damaged.",
        "return_request",
        "I'm sorry your shoes arrived damaged. Could you provide your order ID and briefly describe the damage so I can guide you through the return process?",
        True,
        "G012"
    ),


    # 5. PRODUCT INFORMATION

    (
        "Does this laptop support 32GB of RAM?",
        "product_information",
        "I'd be happy to help you check its memory compatibility. Could you provide the laptop's exact model number?",
        True,
        "G013"
    ),

    (
        "What colors are available for this product?",
        "product_information",
        "I'd be happy to help you check the available colors. Could you share the product name or model?",
        True,
        "G014"
    ),

    (
        "Is this phone compatible with wireless charging?",
        "product_information",
        "Could you share the exact phone model so I can help you check its wireless charging compatibility?",
        True,
        "G015"
    ),


    # 6. TECHNICAL ISSUE

    (
        "Your mobile app crashes every time I open it.",
        "technical_issue",
        "I'm sorry you're experiencing this issue. Which device and operating system are you using?",
        True,
        "G016"
    ),

    (
        "I cannot log into my account even though my password is correct.",
        "technical_issue",
        "I'm sorry you're having trouble logging in. Are you seeing any error message when you try to sign in?",
        True,
        "G017"
    ),

    (
        "The checkout page keeps showing an error when I try to pay.",
        "technical_issue",
        "I'm sorry the checkout page isn't working. Could you share the error message you're seeing?",
        True,
        "G018"
    ),


    # 7. BILLING ISSUE

    (
        "I was charged twice for the same order.",
        "billing_issue",
        "I'm sorry about the duplicate charge. Could you provide your order ID so I can guide you through the appropriate billing support steps?",
        True,
        "G019"
    ),

    (
        "I don't recognize this charge on my account.",
        "billing_issue",
        "I understand your concern. Could you provide the transaction date and amount? Please don't share your full card number or security code.",
        True,
        "G020"
    ),

    (
        "I need a copy of my payment invoice.",
        "billing_issue",
        "I'd be happy to help you find your payment invoice. Could you provide your order ID so I can guide you through the process?",
        True,
        "G021"
    ),


    # 8. COMPLAINT

    (
        "Your customer service is terrible. Nobody has helped me.",
        "complaint",
        "I'm sorry about your experience. Could you tell me what issue you're trying to resolve so I can better understand your situation?",
        True,
        "G022"
    ),

    (
        "I'm extremely disappointed with the quality of your service.",
        "complaint",
        "I'm sorry we haven't met your expectations. Could you tell me more about what went wrong?",
        True,
        "G023"
    ),

    (
        "This is the third time I've contacted support and nobody has resolved my problem.",
        "complaint",
        "I apologize for the repeated inconvenience. Could you briefly describe the unresolved issue and share any existing support reference number? I can help you understand the next steps.",
        True,
        "G024"
    ),


    # 9. OTHER

    (
        "Hello!",
        "other",
        "Hello! How can I help you today?",
        False,
        "G025"
    ),

    (
        "Thank you for your help.",
        "other",
        "You're welcome! Please let me know if you need anything else.",
        False,
        "G026"
    ),

    (
        "Can you explain what your company does?",
        "other",
        "I'd be happy to explain our company. Are you interested in our products, services, or how our business operates?",
        True,
        "G027"
    )

]

In [63]:
seed_samples = [
    make_sample(
        user_message=user,
        intent=intent,
        response=response,
        needs_clarification=clarification,
        group_id=group_id
    )
    for user, intent, response, clarification, group_id in seed_data
]

print(f"Total samples: {len(seed_samples)}")

Total samples: 27


In [64]:
from collections import Counter

intent_distribution = Counter(
    json.loads(sample["messages"][2]["content"])["intent"]
    for sample in seed_samples
)

for intent, count in sorted(intent_distribution.items()):
  print(f"{intent:22} : {count}")

billing_issue          : 3
complaint              : 3
order_status           : 3
other                  : 3
product_information    : 3
refund_request         : 3
return_request         : 3
shipping_issue         : 3
technical_issue        : 3


In [65]:
def validate_dataset(dataset):
  assert len(dataset) > 0

  for index, sample in enumerate(dataset):

    try:
      validate_sample(sample)

      user_message = sample["messages"][1]["content"]
      output = json.loads(sample["messages"][2]["content"])

      assert user_message.strip()
      assert output["response"].strip()

    except Exception as error:
      print(f"Invalid sample at index {index}: {error}")
      raise
  print(f"Dataset validation passed: {len(dataset)} samples")

validate_dataset(seed_samples)

Dataset validation passed: 27 samples


In [66]:
augmentation_data = [

    # ORDER STATUS

    (
        "I'd like to know the current status of my purchase.",
        "order_status",
        "I'd be happy to help you check your order. Could you provide your order ID so I can guide you through the tracking process?",
        True,
        "G002"
    ),

    (
        "Can you track my order number 78921?",
        "order_status",
        "I can help you with your tracking inquiry. I don't have direct access to live tracking information, but you can use your order number on our order tracking page. If you have a separate tracking number, you can use that as well.",
        False,
        "G028"
    ),


    # SHIPPING ISSUE

    (
        "My delivery hasn't arrived on the expected date.",
        "shipping_issue",
        "I'm sorry your delivery is delayed. Could you provide your order ID so I can guide you through the available shipping support options?",
        True,
        "G004"
    ),

    (
        "The courier hasn't delivered my package yet.",
        "shipping_issue",
        "I'm sorry your package hasn't arrived. Could you share your order ID so I can help you understand the next steps for checking its delivery status?",
        True,
        "G029"
    ),


    # REFUND REQUEST

    (
        "How can I get my money back?",
        "refund_request",
        "I can help you understand the refund process. Could you provide your order ID so I can guide you through the next steps?",
        True,
        "G007"
    ),

    (
        "I'd like to cancel my purchase and receive a refund.",
        "refund_request",
        "I can help you understand the cancellation and refund process. Could you provide your order ID so I can guide you through the available options?",
        True,
        "G030"
    ),


    # RETURN REQUEST

    (
        "The shoes don't fit me. How can I return them?",
        "return_request",
        "I'm sorry the shoes don't fit. Could you provide your order ID so I can guide you through the return process?",
        True,
        "G011"
    ),

    (
        "Can I send this product back?",
        "return_request",
        "I'd be happy to help you understand the return process. Could you share the product name and your order ID?",
        True,
        "G031"
    ),


    # PRODUCT INFORMATION

    (
        "What are the dimensions of this product?",
        "product_information",
        "I can help you find the product dimensions. Could you share the product name or model?",
        True,
        "G032"
    ),

    (
        "Do you have this item in other colors?",
        "product_information",
        "I'd be happy to help you check the available colors. Could you share the product name or model?",
        True,
        "G014"
    ),


    # TECHNICAL ISSUE

    (
        "The website stops responding whenever I click checkout.",
        "technical_issue",
        "I'm sorry you're experiencing this issue. Which browser are you using, and do you see any error message?",
        True,
        "G018"
    ),

    (
        "I keep getting an error when trying to reset my password.",
        "technical_issue",
        "I'm sorry you're having trouble resetting your password. Could you share the error message you're seeing?",
        True,
        "G033"
    ),


    # BILLING ISSUE

    (
        "Why was I charged more than the displayed price?",
        "billing_issue",
        "I understand your concern. Could you provide your order ID and the amount you were charged? Please don't share your full card number or security code.",
        True,
        "G034"
    ),

    (
        "My payment was deducted, but the order still shows as unpaid.",
        "billing_issue",
        "I'm sorry about this payment issue. Could you provide your order ID and payment transaction reference? Please don't share your full card number or security code.",
        True,
        "G035"
    ),


    # COMPLAINT

    (
        "I'm really unhappy with the way your team handled my issue.",
        "complaint",
        "I'm sorry about your experience. Could you tell me what happened and which issue you're still having trouble with?",
        True,
        "G022"
    ),

    (
        "This is unacceptable. I've had a terrible experience with your service.",
        "complaint",
        "I'm sorry we haven't met your expectations. Could you tell me what went wrong so I can better understand your concerns?",
        True,
        "G023"
    ),


    # OTHER

    (
        "Do you offer any loyalty programs?",
        "other",
        "I'd be happy to help you find information about our loyalty programs. Are you interested in program eligibility, earning rewards, or redeeming points?",
        True,
        "G036"
    ),

    (
        "Where can I find your company information?",
        "other",
        "I can help you find company information. Are you looking for our company background, contact details, or information about our products and services?",
        True,
        "G037"
    )

]

In [67]:
agumented_samples = [
    make_sample(
        user_message=user,
        intent=intent,
        response=response,
        needs_clarification=clarification,
        group_id=group_id
    )
    for user, intent, response, clarification, group_id in augmentation_data
]

In [68]:
print("Original samples:", len(seed_samples))
print("Agumented samples:", len(agumented_samples))

dataset = seed_samples + agumented_samples

assert all("group_id" in sample for sample in dataset)
print("All samples have group_id!")

print("Total samples:", len(dataset))

Original samples: 27
Agumented samples: 18
All samples have group_id!
Total samples: 45


In [69]:
from collections import Counter

group_distribution = Counter(
    sample["group_id"] for sample in dataset
)

print("Total samples:", len(dataset))
print("Unique groups:", len(group_distribution))

for group_id, count in sorted(group_distribution.items()):
    if count > 1:
        print(f"{group_id}: {count} samples")

Total samples: 45
Unique groups: 37
G002: 2 samples
G004: 2 samples
G007: 2 samples
G011: 2 samples
G014: 2 samples
G018: 2 samples
G022: 2 samples
G023: 2 samples


In [70]:
from collections import Counter
import json

intent_distribution = Counter(
    json.loads(sample["messages"][2]["content"])["intent"]
    for sample in dataset
)

for intent in INTENTS:
  print(f"{intent:22}: {intent_distribution[intent]}")

order_status          : 5
shipping_issue        : 5
refund_request        : 5
return_request        : 5
product_information   : 5
technical_issue       : 5
billing_issue         : 5
complaint             : 5
other                 : 5


In [71]:
def find_duplicates(dataset):

  seen = set()
  duplicates = []

  for index, sample in enumerate(dataset):
    user_message = sample["messages"][1]["content"]

    if user_message in seen:
      duplicates.append(index)

    seen.add(user_message)

  return duplicates


duplicates = find_duplicates(dataset)

print("Duplicates samples:", len(duplicates))

Duplicates samples: 0


In [72]:
validate_dataset(dataset)

Dataset validation passed: 45 samples


In [73]:
clarification_distribution = Counter(
    json.loads(sample["messages"][2]["content"])["needs_clarification"]
    for sample in dataset
)

print("Needs clarification:", clarification_distribution[True])
print("No clarification:", clarification_distribution[False])

Needs clarification: 41
No clarification: 4


In [74]:
message_lengths = [
    len(sample["messages"][1]["content"].split())
    for sample in dataset
]

print("Minimum:", min(message_lengths))
print("Maximum:", max(message_lengths))
print("Average:", sum(message_lengths) / len(message_lengths))

Minimum: 1
Maximum: 14
Average: 8.666666666666666


In [75]:
for index, sample in enumerate(dataset):
  message = sample["messages"][1]["content"]
  word_count = len(message.split())

  if word_count < 3 or word_count > 40:
    print(f"Index: {index}")
    print(f"Words: {word_count}")
    print(f"Message: {message}")
    print("-" * 50)

Index: 24
Words: 1
Message: Hello!
--------------------------------------------------


In [76]:
duplicates = find_duplicates(dataset)

print("Duplicate indices:", duplicates)
print("Duplicate count:", len(duplicates))

Duplicate indices: []
Duplicate count: 0


In [77]:
!pip install -q scikit-learn

In [78]:
from sklearn.model_selection import GroupShuffleSplit

groups = [sample["group_id"] for sample in dataset]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_val_indices, test_indices = next(
    splitter.split(dataset, groups=groups)
)

train_val_data = [dataset[i] for i in train_val_indices]
test_data = [dataset[i] for i in test_indices]

In [79]:
train_val_groups = [
    sample["group_id"]
    for sample in train_val_data
]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.125,
    random_state=42
)

train_indices, val_indices = next(
    splitter.split(train_val_data, groups=train_val_groups)
)

train_data = [train_val_data[i] for i in train_indices]
val_data = [train_val_data[i] for i in val_indices]

In [80]:
train_groups = {s["group_id"] for s in train_data}
val_groups = {s["group_id"] for s in val_data}
test_groups = {s["group_id"] for s in test_data}

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("No group leakage detected!")

No group leakage detected!


### 4. Tokenization & Chat Templates

In [ ]:
# Check tokenizer chat template

assert tokenizer.chat_template is not None

print("Chat template is available.")

Chat template is available.


In [ ]:
def format_sample(sample):

  text = tokenizer.apply_chat_template(
      sample["messages"],
      tokenize=False,
      add_generation_prompt=False,
      enable_thinking=False
  )


  return {
      "group_id": sample["group_id"],
      "text": text
  }

In [ ]:
train_formatted = [
    format_sample(sample)
    for sample in train_data
]

val_formatted = [
    format_sample(sample)
    for sample in val_data
]

test_formatted = [
    format_sample(sample)
    for sample in test_data
]

In [ ]:
print(train_formatted)

[{'group_id': 'G001', 'text': '<|im_start|>system\n\nYou are SupportMind, a professional customer support assistant.\n\nYour responsibilities:\n1. Identify the customer\'s primary intent.\n2. Provide a helpful and professional response.\n3. Ask for clarification when required.\n4. Never invent information.\n5. Focus on the customer\'s primary requested action\n   when selecting an intent.\n\nSupported intents:\n- order_status\n- shipping_issue\n- refund_request\n- return_request\n- product_information\n- technical_issue\n- billing_issue\n- complaint\n- other\n\nReturn only a valid JSON object containing exactly:\n- intent: string\n- response: string\n- needs_clarification: boolean\n\nDo not include Markdown or explanations outside JSON.\n<|im_end|>\n<|im_start|>user\nWhere is my order? My order ID is 12345.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n{"intent": "order_status", "response": "I can help you with your order inquiry. Please check your tracking information for t

In [ ]:
print(train_formatted[0]["text"])

<|im_start|>system

You are SupportMind, a professional customer support assistant.

Your responsibilities:
1. Identify the customer's primary intent.
2. Provide a helpful and professional response.
3. Ask for clarification when required.
4. Never invent information.
5. Focus on the customer's primary requested action
   when selecting an intent.

Supported intents:
- order_status
- shipping_issue
- refund_request
- return_request
- product_information
- technical_issue
- billing_issue
- complaint
- other

Return only a valid JSON object containing exactly:
- intent: string
- response: string
- needs_clarification: boolean

Do not include Markdown or explanations outside JSON.
<|im_end|>
<|im_start|>user
Where is my order? My order ID is 12345.<|im_end|>
<|im_start|>assistant
<think>

</think>

{"intent": "order_status", "response": "I can help you with your order inquiry. Please check your tracking information for the latest status.", "needs_clarification": false}<|im_end|>



In [ ]:
def tokenize_samples(samples):

  tokenized_samples = []

  for sample in samples:
      encoded = tokenizer(
        sample["text"],
        add_special_tokens=False,
        truncation=False
      )

      tokenized_samples.append({
          "group_id": sample["group_id"],
          "input_ids": encoded["input_ids"],
          "attention_mask": encoded["attention_mask"]
      })

  return tokenized_samples

In [ ]:
train_tokenized = tokenize_samples(train_formatted)

val_tokenized = tokenize_samples(val_formatted)

test_tokenized = tokenize_samples(test_formatted)

In [ ]:
print(train_tokenized[0])

{'group_id': 'G001', 'input_ids': [151644, 8948, 271, 2610, 525, 9186, 69638, 11, 264, 6584, 6002, 1824, 17847, 382, 7771, 27323, 510, 16, 13, 64547, 279, 6002, 594, 6028, 7385, 624, 17, 13, 39565, 264, 10950, 323, 6584, 2033, 624, 18, 13, 20437, 369, 63684, 979, 2567, 624, 19, 13, 14695, 17023, 1995, 624, 20, 13, 25806, 389, 279, 6002, 594, 6028, 11223, 1917, 198, 256, 979, 26301, 458, 7385, 382, 34636, 93515, 510, 12, 1973, 4773, 198, 12, 11601, 53340, 198, 12, 20965, 7893, 198, 12, 470, 7893, 198, 12, 1985, 35212, 198, 12, 10916, 53340, 198, 12, 33531, 53340, 198, 12, 12181, 198, 12, 1008, 271, 5598, 1172, 264, 2697, 4718, 1633, 8482, 6896, 510, 12, 7385, 25, 914, 198, 12, 2033, 25, 914, 198, 12, 3880, 6794, 277, 2404, 25, 2710, 271, 5404, 537, 2924, 73192, 476, 40841, 4889, 4718, 624, 151645, 198, 151644, 872, 198, 9064, 374, 847, 1973, 30, 3017, 1973, 3034, 374, 220, 16, 17, 18, 19, 20, 13, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 4913, 56431, 788, 330, 1358, 477

In [ ]:
import numpy as np

def analyze_token_lengths(samples, name):

  lengths = [
      len(sample["input_ids"])
      for sample in samples
  ]

  print(f"\n{name}")
  print("-" * 30)
  print("Samples:", len(lengths))
  print("Min:", min(lengths))
  print("Max:", max(lengths))
  print("Mean:", round(np.mean(lengths), 2))
  print("Median:", np.median(lengths))

analyze_token_lengths(train_tokenized, "Train")
analyze_token_lengths(val_tokenized, "Validation")
analyze_token_lengths(test_tokenized, "Test")


Train
------------------------------
Samples: 30
Min: 184
Max: 220
Mean: 204.63
Median: 204.0

Validation
------------------------------
Samples: 4
Min: 200
Max: 206
Mean: 202.5
Median: 202.0

Test
------------------------------
Samples: 11
Min: 193
Max: 208
Mean: 202.0
Median: 204.0


In [ ]:
def validate_tokenized_dataset(samples, name):

    errors = []

    for index, sample in enumerate(samples):

        if len(sample["input_ids"]) != len(sample["attention_mask"]):
            errors.append(f"Sample {index}: Length mismatch")

        if len(sample["input_ids"]) == 0:
            errors.append(f"Sample {index}: Empty input_ids")

        if not all(
            mask in (0, 1)
            for mask in sample["attention_mask"]
        ):
            errors.append(f"Sample {index}: Invalid attention_mask")

    print(f"\n{name}")
    print("Total samples:", len(samples))
    print("Errors:", len(errors))

    for error in errors:
        print(error)

    return len(errors) == 0

In [ ]:
train_valid = validate_tokenized_dataset(
    train_tokenized, "Train"
)

val_valid = validate_tokenized_dataset(
    val_tokenized, "Validation"
)

test_valid = validate_tokenized_dataset(
    test_tokenized, "Test"
)

assert train_valid and val_valid and test_valid


Train
Total samples: 30
Errors: 0

Validation
Total samples: 4
Errors: 0

Test
Total samples: 11
Errors: 0


In [ ]:
sample = train_data[0]

probe = tokenizer.apply_chat_template(
    sample["messages"],
    tokenize=True,
    add_generation_prompt=False,
    return_dict=True,
    return_assistant_tokens_mask=True,
    enable_thinking=False
)

print("Available keys:", probe.keys())

if "assistant_masks" in probe:
    print("Assistant mask is available.")

    print("Input IDs:", len(probe["input_ids"]))
    print("Assistant mask:", len(probe["assistant_masks"]))

    print(
        "Assistant tokens:",
        sum(probe["assistant_masks"])
    )
else:
    print("Assistant mask is not available.")

[transformers] return_assistant_tokens_mask==True but chat template does not contain `{% generation %}` keyword.


Available keys: KeysView({'input_ids': [151644, 8948, 271, 2610, 525, 9186, 69638, 11, 264, 6584, 6002, 1824, 17847, 382, 7771, 27323, 510, 16, 13, 64547, 279, 6002, 594, 6028, 7385, 624, 17, 13, 39565, 264, 10950, 323, 6584, 2033, 624, 18, 13, 20437, 369, 63684, 979, 2567, 624, 19, 13, 14695, 17023, 1995, 624, 20, 13, 25806, 389, 279, 6002, 594, 6028, 11223, 1917, 198, 256, 979, 26301, 458, 7385, 382, 34636, 93515, 510, 12, 1973, 4773, 198, 12, 11601, 53340, 198, 12, 20965, 7893, 198, 12, 470, 7893, 198, 12, 1985, 35212, 198, 12, 10916, 53340, 198, 12, 33531, 53340, 198, 12, 12181, 198, 12, 1008, 271, 5598, 1172, 264, 2697, 4718, 1633, 8482, 6896, 510, 12, 7385, 25, 914, 198, 12, 2033, 25, 914, 198, 12, 3880, 6794, 277, 2404, 25, 2710, 271, 5404, 537, 2924, 73192, 476, 40841, 4889, 4718, 624, 151645, 198, 151644, 872, 198, 9064, 374, 847, 1973, 30, 3017, 1973, 3034, 374, 220, 16, 17, 18, 19, 20, 13, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 4913, 56431, 788, 330, 1358

### 5. Model Loading & Precision Analysis

In [ ]:
from transformers import AutoConfig

MODEL_NAME = "Qwen/Qwen3-4B"

model_config = AutoConfig.from_pretrained(MODEL_NAME)

print("Model architecture:", model_config.architectures)
print("Model type:", model_config.model_type)
print("Hidden size:", model_config.hidden_size)
print("Layers:", model_config.num_hidden_layers)
print("Attention heads:", model_config.num_attention_heads)
print("KV heads:", model_config.num_key_value_heads)
print("Vocabulary size:", model_config.vocab_size)
print("Max positions:", model_config.max_position_embeddings)

Model architecture: ['Qwen3ForCausalLM']
Model type: qwen3
Hidden size: 2560
Layers: 36
Attention heads: 32
KV heads: 8
Vocabulary size: 151936
Max positions: 40960


In [ ]:
import torch

if torch.cuda.is_available():

    gpu = torch.cuda.get_device_properties(0)

    print("GPU:", gpu.name)
    print("Total VRAM:", round(gpu.total_memory / 1024**3, 2), "GB")
    print("CUDA Capability:", torch.cuda.get_device_capability(0))

    print("FP16 supported:", torch.cuda.is_available())

    print(
        "BF16 supported:",
        torch.cuda.is_bf16_supported()
    )

else:
    print("CUDA is not available.")

CUDA is not available.


In [ ]:
PARAM_COUNT = 4_000_000_000

PRECISIONS = {
    "FP32": 4,
    "FP16": 2,
    "INT8": 1,
    "4-bit": 0.5
}

for precision, bytes_per_param in PRECISIONS.items():
  memory_bytes = PARAM_COUNT * bytes_per_param
  memory_gib = memory_bytes / (1024 ** 3)

  print(f"{precision}: {memory_gib:.2f} GIB")

FP32: 14.90 GIB
FP16: 7.45 GIB
INT8: 3.73 GIB
4-bit: 1.86 GIB


In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=False,
    load_in_8bit=False,
    load_in_16bit=True
)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=False,
    load_in_8bit=True,
    load_in_16bit=False
)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=True,
    load_in_8bit=False,
    load_in_16bit=False
)

##### Memory Footprint

In [ ]:
memory_bytes = model.get_memory_footprint()

print(
    "Model memory:",
    round(memory_bytes / (1024 ** 3), 2),
    "GiB"
)

In [ ]:
import torch
import inspect
import unsloth

from unsloth import FastLanguageModel

print("Unsloth version:", unsloth.__version__)
print("PyTorch version:", torch.__version__)

print("\nFastLanguageModel:")
print(FastLanguageModel)

print("\nfrom_pretrained signature:")
print(inspect.signature(FastLanguageModel.from_pretrained))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1118 14:19:58.499000 495 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1118 14:19:58.547000 495 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth version: 2026.11.6
PyTorch version: 2.14.1+cu130

FastLanguageModel:
<class 'unsloth.models.loader.FastLanguageModel'>

from_pretrained signature:
(model_name='unsloth/Llama-3.2-1B-Instruct', max_seq_length=2048, dtype=None, load_in_4bit=True, load_in_8bit=False, load_in_16bit=False, full_finetuning=False, token=None, device_map='sequential', device_map_planner_kwargs=None, rope_scaling=None, fix_tokenizer=True, trust_remote_code=False, use_gradient_checkpointing='unsloth', resize_model_vocab=None, revision=None, use_exact_model_name=False, offload_embedding='auto', float32_mixed_precision=None, fast_inference=False, gpu_memory_utilization=0.5, flo

### 6. Quantization Experiments (FP16 → INT8 → 4-bit)

In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print(bnb_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [ ]:
PARAM_COUNT = 4_000_000_000

precisions = {
    "FP32": 32,
    "FP16": 16,
    "INT8": 8,
    "NF4": 4
}

for name, bits in precisions.items():

    memory_bytes = PARAM_COUNT * bits / 8

    memory_gib = memory_bytes / (1024 ** 3)

    print(f"{name}: {memory_gib:.2f} GiB")

FP32: 14.90 GiB
FP16: 7.45 GiB
INT8: 3.73 GiB
NF4: 1.86 GiB


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=False
)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=True
)

### 7. LoRA Configuration


 class LoRa:

    def __init__(
      self, r: int,
      alpha: int,
      W: np.ndarray
    ) -> None:

      self.r = r
      self.alpha = alpha
      self.W = W
      self.out_features,
      self.in_features = W.shape
      self.A , self.B = self.rank_maker()
    
    def forward_pass(
      self,
      x: np.array
    ) -> np.array:

      base_output = self.W @ x
      lora_output = self.B @ (self.A @ x)
      scaling = self.alpha / self.r
      return base_output + scaling * lora_output

    
    def rank_maker(self) -> np.array:
      n = self.W.shape[0]
      A = np.random.norma;(0, 0.01, (self.r, self.in_features))
      B = np.zeros((self.out_features, self.r))
      return A, B

In [ ]:
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,

    "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],

    "bias": "none",
    "use_gradient_checkpointing": "unsloth",
    "use_rslora": False
}

for key, value in LORA_CONFIG.items():
  print(f"{key}: {value}")

r: 16
lora_alpha: 32
lora_dropout: 0.0
target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
bias: none
use_gradient_checkpointing: unsloth
use_rslora: False


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,

    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",

    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False
)

##### Trainable Parameters

In [ ]:
model.print_trainable_parameters()

### 8. SFT Dataset Preparation

In [29]:
!pip install -U datasets

  Using cached datasets-5.2.0-py3-none-any.whl.metadata (24 kB)
Using cached datasets-5.2.0-py3-none-any.whl (608 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.11.8 requires datasets!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0,<5.0.0,>=3.4.1, but you have datasets 5.2.0 which is incompatible.
unsloth 2026.11.9 requires datasets!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0,<5.0.0,>=3.4.1, but you have datasets 5.2.0 which is incompatible.


In [81]:
from datasets import Dataset

def prepare_sft_dataset(samples):

    formatted_samples = [
        {
            "messages": sample["messages"]
        }
        for sample in samples
    ]

    return Dataset.from_list(formatted_samples)

In [82]:
train_sft_dataset = prepare_sft_dataset(train_data)

val_sft_dataset = prepare_sft_dataset(val_data)

test_sft_dataset = prepare_sft_dataset(test_data)

In [83]:
print("Train:", len(train_sft_dataset))
print("Validation:", len(val_sft_dataset))
print("Test:", len(test_sft_dataset))

print("\nDataset Features:")
print(train_sft_dataset.features)

print("\nFirst Sample:")
print(train_sft_dataset[0])

Train: 30
Validation: 4
Test: 11

Dataset Features:
{'messages': List({'role': Value('string'), 'content': Value('string')})}

First Sample:
{'messages': [{'role': 'system', 'content': "\nYou are SupportMind, a professional customer support assistant.\n\nYour responsibilities:\n1. Identify the customer's primary intent.\n2. Provide a helpful and professional response.\n3. Ask for clarification when required.\n4. Never invent information.\n5. Focus on the customer's primary requested action\n   when selecting an intent.\n\nSupported intents:\n- order_status\n- shipping_issue\n- refund_request\n- return_request\n- product_information\n- technical_issue\n- billing_issue\n- complaint\n- other\n\nReturn only a valid JSON object containing exactly:\n- intent: string\n- response: string\n- needs_clarification: boolean\n\nDo not include Markdown or explanations outside JSON.\n"}, {'role': 'user', 'content': 'Where is my order? My order ID is 12345.'}, {'role': 'assistant', 'content': '{"intent

In [84]:
!pip install -U trl

In [85]:
import inspect

from trl import SFTTrainer, SFTConfig

print("SFTTrainer Signature:")
print(inspect.signature(SFTTrainer.__init__))

print("\nSFTConfig Signature:")
print(inspect.signature(SFTConfig))

SFTTrainer Signature:
(self, model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callba

In [86]:
import trl

print("TRL Version:", trl.__version__)
print("Transformers Version:", __import__("transformers").__version__)

TRL Version: 1.18.2
Transformers Version: 5.17.0


In [87]:
train_groups = {s["group_id"] for s in train_data}
val_groups = {s["group_id"] for s in val_data}
test_groups = {x["group_id"] for x in test_data}

print("Train ∩ Validation:", train_groups & val_groups)
print("Train ∩ Test:", train_groups & test_groups)
print("Validation ∩ Test:", val_groups & test_groups)

Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()


In [88]:
from trl import SFTConfig

SFT_CONFIG = {
    "output_dir": "./outputs/supportmind",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "warmup_steps": 1,
    "lr_scheduler_type": "cosine",
    "logging_steps": 1,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "max_length": 512,
    "packing": False,
    "assistant_only_loss": True,
    "report_to": "none",
    "seed": 3407
}

sft_config = SFTConfig(**SFT_CONFIG)

In [89]:
print(sft_config)

SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=True,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=Fa

##### Data Quality Audit

In [90]:
import json

def validate_sample(sample):
  assistant_message = sample["messages"][-1]["content"]

  try:
    response = json.loads(assistant_message)
  except json.JSONDecoderError:
    return False, "Invalid JSON"

  required_fields = {
      "intent",
      "response",
      "needs_clarification"
  }

  if set(response.keys()) != required_fields:
    return False, "Invakid fields"

  if response["intent"] not in INTENTS:
    return False, "Invalid intent"

  if not isinstance(response["response"], str):
    return False, "Invalid response type"

  if not isinstance(response["needs_clarification"], bool):
        return False, "Invalid clarification type"

  return True, "Valid"


for split_name, dataset in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:

  results = [validate_sample(x) for x in dataset]

  valid_count = sum(valid for valid, _ in results)
  invalid_count = len(results) - valid_count

  print(f"\n{split_name}")
  print("Valid:", valid_count)
  print("Invalid:", invalid_count)

  for index, (valid, reason) in enumerate(results):
    if not valid:
      print(f"Sample {index}: {reason}")


Train
Valid: 30
Invalid: 0

Validation
Valid: 4
Invalid: 0

Test
Valid: 11
Invalid: 0


##### Semantic Quality Audit

In [91]:
for split_name, dataset in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    print(f"\n{'=' * 20} {split_name} {'=' * 20}")

    for index, sample in enumerate(dataset):
      user_message = sample["messages"][1]["content"]
      assistant_response = json.loads(
          sample["messages"][2]["content"]
      )

      print(f"\nSample {index + 1}")
      print("User:", user_message)
      print("Intent:", assistant_response["intent"])
      print("Response:", assistant_response["response"])
      print(
          "Needs Clarification:",
          assistant_response["needs_clarification"]
      )


==================== Train ====================

Sample 1
User: Where is my order? My order ID is 12345.
Intent: order_status
Response: I can help you with your order inquiry. I don't have direct access to live order information, but you can check the latest status using your order ID on our order tracking page.
Needs Clarification: False

Sample 2
User: Can you tell me where my package is?
Intent: order_status
Response: I'd be happy to help you check your order. Could you provide your order ID so I can guide you through the tracking process?
Needs Clarification: True

Sample 3
User: I placed an order three days ago, but I haven't received any updates.
Intent: order_status
Response: I understand your concern. Could you share your order ID so I can help you find the appropriate way to check its status?
Needs Clarification: True

Sample 4
User: My package is five days late.
Intent: shipping_issue
Response: I'm sorry your package is delayed. Could you provide your order ID so I can guide

### 9. Supervised Fine-Tuning

### 10. Training Analysis

### 11. Model Evaluation

### 12. Adapter Management & Export

### 13. Final Inference